In [ ]:
"""
locate_surface_events_2025.py
-------------------------------
Scans all daily common-detection CSVs for 2025, filters for:
  - pure "su" events (every station in all_classes == 'su')
  - detected at > 8 stations

For each qualifying event:
  1. Downloads vertical-component waveforms from IRIS FDSN
  2. Preprocesses: demean → taper → resample 50 Hz → bandpass 1–8 Hz → SNR filter
  3. Computes Hilbert envelopes → downsample 5 Hz → lowpass 0.2 Hz
  4. Attaches station coordinates
  5. Runs enveloc (XCOR) to locate
  6. Collects result into a single output CSV

Usage:
    conda activate enveloc   # or whichever env has enveloc installed
    python locate_surface_events_2025.py

Edit the CONFIG block below to match your paths.
"""

import os
import ast
import glob
import logging
import traceback
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.signal import hilbert

from obspy import Stream, UTCDateTime
from obspy.clients.fdsn import Client
from obspy.core.util import AttribDict
from obspy.signal.filter import envelope

from enveloc.core import XCOR

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG — edit these paths / parameters to match your setup
# ─────────────────────────────────────────────────────────────────────────────

# Directory that holds the daily common-detection CSVs
#   Expected glob pattern: common_20250101_0000_to_20250101_2359_events.csv
CSV_DIR = "../logs/mt_rainier_common_detections"

# Output CSV path
OUTPUT_CSV = "../logs/surface_events_enveloc_located_2025.csv"

# Waveform download window (seconds) around rounded_start
SECONDS_BEFORE = 0
SECONDS_AFTER  = 200

# Preprocessing parameters (must match notebook)
FREQMIN  = 1.0    # bandpass low  (Hz)
FREQMAX  = 8.0    # bandpass high (Hz)
LOWPASS  = 0.2    # envelope smoothing lowpass (Hz)
DT       = 10     # edge trim after taper (seconds each side)
TARGET_FS_WAVEFORM = 50.0   # resample target for filtered waveforms
TARGET_FS_ENVELOPE = 5.0    # resample target for envelopes
SNR_THRESHOLD = 5.0          # minimum robust envelope SNR to keep a trace

# Station filter: only events seen at strictly more than this many stations
MIN_STATIONS = 10

# Geographic bounding box for FDSN station queries (Mount Rainier region)
MINLAT, MAXLAT = 46.0, 49.0
MINLON, MAXLON = -123.0, -120.0

# FDSN client
FDSN_CLIENT = "IRIS"

# Enveloc grid over the PNW region (uncomment to use a fixed grid instead of
# the auto-generated one; fixed grid is more reproducible)
ENVELOC_GRID = {
    "deps": np.arange(0.1, 1.0, 0.5),
    "lons": np.arange(-123.0, -120.0, 0.05),
    "lats": np.arange(46.0, 49.0, 0.05),
}

# ─────────────────────────────────────────────────────────────────────────────
# Logging
# ─────────────────────────────────────────────────────────────────────────────

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
log = logging.getLogger(__name__)

# ─────────────────────────────────────────────────────────────────────────────
# Helper utilities (ported directly from the notebook)
# ─────────────────────────────────────────────────────────────────────────────

def is_pure_su(all_classes_str: str) -> bool:
    """Return True only if every classification in all_classes is 'su'."""
    try:
        classes = ast.literal_eval(all_classes_str)
    except Exception:
        return False
    return bool(classes) and all(c.strip() == "su" for c in classes)


def drop_stations_with_gaps(st: Stream, max_total_gap_s: float = 0.0) -> Stream:
    """
    Merge per-trace-id fragments. Drop any trace whose total gap length
    exceeds max_total_gap_s (default 0 = zero tolerance).
    """
    out = Stream()
    for tr_id in sorted(set(tr.id for tr in st)):
        s = st.select(id=tr_id).copy()
        if not s:
            continue
        gaps = s.get_gaps()
        total_gap = sum(float(g[6]) for g in gaps) if gaps else 0.0
        if total_gap > max_total_gap_s:
            log.debug("  Dropped %s (gap %.2f s)", tr_id, total_gap)
            continue
        s.merge(method=1)
        out += s[0]
    return out


def resolve_net_loc_chan(client: Client, station: str,
                         t_start: UTCDateTime, t_end: UTCDateTime,
                         loc_preference=("01", "", "--"),
                         chan_hint: str = "*HZ"):
    """
    Query IRIS for network/location/channel metadata for a station name,
    inside the Mount Rainier bounding box, preferring loc codes in order.
    Returns (net, sta, loc, chan) tuple or None if not found.
    """
    try:
        inv = client.get_stations(
            network="*", station=station, channel=chan_hint,
            starttime=t_start, endtime=t_end,
            minlatitude=MINLAT, maxlatitude=MAXLAT,
            minlongitude=MINLON, maxlongitude=MAXLON,
            level="channel",
        )
    except Exception:
        return None

    candidates = []
    for net in inv:
        for sta in net:
            for cha in sta:
                loc = cha.location_code or ""
                candidates.append((net.code, sta.code, loc, cha.code))

    if not candidates:
        return None

    preferred = [c for c in candidates if c[2] in loc_preference]
    pool = preferred if preferred else candidates
    loc_rank = {loc: i for i, loc in enumerate(loc_preference)}
    pool.sort(key=lambda x: (loc_rank.get(x[2], 999), x[3]))
    return pool[0]


def download_best_vertical(client: Client, station: str,
                            t_start: UTCDateTime, t_end: UTCDateTime) -> Stream:
    """Download BHZ → HHZ → EHZ for a station in the bounding box."""
    resolved = resolve_net_loc_chan(client, station, t_start, t_end)
    if resolved is None:
        log.debug("  [%s] not found in region/time window", station)
        return Stream()

    net, sta, loc, _ = resolved
    for ch in ("BHZ", "HHZ", "EHZ"):
        try:
            st = client.get_waveforms(net, sta, loc, ch, t_start, t_end)
            if st:
                log.debug("  Downloaded %s.%s.%s.%s", net, sta, loc, ch)
                return st
        except Exception:
            pass

    log.debug("  [%s] no BHZ/HHZ/EHZ for %s.%s.%s", station, net, sta, loc)
    return Stream()


def download_event_stream(row: pd.Series, client: Client) -> Stream:
    """Download waveforms for all stations associated with a detection row."""
    start_time = UTCDateTime(pd.Timestamp(row["rounded_start"]).to_pydatetime())
    t_start = start_time - SECONDS_BEFORE
    t_end   = start_time + SECONDS_AFTER

    stations = ast.literal_eval(row["stations"])
    stream = Stream()
    for sta in stations:
        stream += download_best_vertical(client, sta, t_start, t_end)
    return stream


def robust_envelope_snr(tr, q: float = 0.99, eps: float = 1e-12) -> float:
    """SNR = quantile(env, q) / median(env) using Hilbert envelope."""
    x = np.asarray(tr.data, dtype=float)
    if x.size == 0:
        return np.nan
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    env = np.abs(hilbert(x))
    med = np.median(env)
    if not np.isfinite(med) or med < eps:
        return np.nan
    return float(np.quantile(env, q) / med)


def filter_low_snr_traces(st: Stream, threshold: float = SNR_THRESHOLD):
    """Drop traces below the robust envelope SNR threshold."""
    keep = []
    for tr in st:
        snr = robust_envelope_snr(tr)
        if np.isfinite(snr) and snr >= threshold:
            keep.append(tr)
    dropped = len(st) - len(keep)
    if dropped:
        log.debug("  SNR filter: kept %d / %d traces", len(keep), len(st))
    return Stream(keep)


def attach_coordinates(st_env: Stream, client: Client) -> None:
    """
    Query FDSN for each trace's station coordinates and attach them
    in-place as tr.stats.coordinates (AttribDict with lat/lon/elevation).
    Traces that fail the coordinate lookup are left without coordinates.
    """
    for tr in st_env:
        try:
            inv = client.get_stations(
                network=tr.stats.network,
                station=tr.stats.station,
                location=tr.stats.location,
                channel=tr.stats.channel,
                starttime=tr.stats.starttime,
                endtime=tr.stats.endtime,
            )
            tr.stats.coordinates = AttribDict({
                "latitude":  inv[0][0].latitude,
                "longitude": inv[0][0].longitude,
                "elevation": inv[0][0].elevation,
            })
        except Exception as e:
            log.debug("  Could not get coords for %s: %s", tr.id, e)


def preprocess_and_locate(stream: Stream, client: Client):
    """
    Full preprocessing pipeline → enveloc location.

    Returns (latitude, longitude) floats, or (nan, nan) on failure.
    Also returns the number of traces that survived preprocessing.
    """
    if not stream:
        return np.nan, np.nan, 0

    # ── 1. Drop gappy traces ───────────────────────────────────────────────
    stream = drop_stations_with_gaps(stream, max_total_gap_s=0.0)
    if not stream:
        log.debug("  No traces survived gap filter.")
        return np.nan, np.nan, 0

    # ── 2. Bandpass filter ────────────────────────────────────────────────
    st_filt = stream.copy()
    st_filt.detrend("demean")
    st_filt.taper(max_percentage=None, max_length=5)
    st_filt.resample(TARGET_FS_WAVEFORM)
    st_filt.filter("bandpass", freqmin=FREQMIN, freqmax=FREQMAX,
                   corners=3, zerophase=True)

    # ── 3. SNR filter ─────────────────────────────────────────────────────
    st_filt = filter_low_snr_traces(st_filt, threshold=SNR_THRESHOLD)
    if not st_filt:
        log.debug("  No traces survived SNR filter.")
        return np.nan, np.nan, 0

    # ── 4. Build envelopes ────────────────────────────────────────────────
    st_env = st_filt.copy()
    for tr in st_env:
        # Hilbert requires even-length arrays
        if tr.stats.npts % 2 == 1:
            tr.trim(tr.stats.starttime,
                    tr.stats.endtime + 1.0 / tr.stats.sampling_rate,
                    pad=True, fill_value=0)
        tr.data = envelope(tr.data)
        tr.resample(TARGET_FS_ENVELOPE)

    st_env.filter("lowpass", freq=LOWPASS)

    # ── 5. Trim taper edges ───────────────────────────────────────────────
    t1 = st_filt[0].stats.starttime + DT
    t2 = st_filt[0].stats.endtime   - DT
    st_filt.trim(t1, t2)
    st_env.trim(t1, t2)

    # ── 6. Attach coordinates ─────────────────────────────────────────────
    attach_coordinates(st_env, client)

    # Drop any traces that didn't get coordinates (enveloc requires them)
    st_env_ok = Stream([tr for tr in st_env
                        if hasattr(tr.stats, "coordinates")])
    if len(st_env_ok) < 3:
        log.debug("  Too few traced with coordinates (%d).", len(st_env_ok))
        return np.nan, np.nan, len(st_env_ok)

    n_traces = len(st_env_ok)

    # ── 7. Run enveloc ────────────────────────────────────────────────────
    try:
        XC = XCOR(st_env_ok, plot=False, interact=False)
        loc = XC.locate()
        lat = float(loc.latitude)  if loc.latitude  is not None else np.nan
        lon = float(loc.longitude) if loc.longitude is not None else np.nan
        return lat, lon, n_traces
    except Exception as e:
        log.debug("  enveloc failed: %s", e)
        return np.nan, np.nan, n_traces


# ─────────────────────────────────────────────────────────────────────────────
# Main
# ─────────────────────────────────────────────────────────────────────────────

def collect_qualifying_events(csv_dir: str) -> pd.DataFrame:
    """
    Walk all daily CSVs in csv_dir, keep rows that are:
      - pure 'su' classification
      - num_stations > MIN_STATIONS
    Returns a combined dataframe with a 'source_file' column added.
    """
    pattern = os.path.join(csv_dir, "common_2025*_events.csv")
    files = sorted(glob.glob(pattern))

    if not files:
        raise FileNotFoundError(
            f"No 2025 CSV files found matching:\n  {pattern}\n"
            "Check that CSV_DIR is correct."
        )

    log.info("Found %d daily CSV files for 2025.", len(files))

    frames = []
    for fpath in files:
        try:
            df = pd.read_csv(fpath)
        except Exception as e:
            log.warning("Could not read %s: %s", fpath, e)
            continue

        # Pure-su filter
        mask_su = df["all_classes"].apply(is_pure_su)
        # Station-count filter
        mask_sta = df["num_stations"] > MIN_STATIONS

        kept = df[mask_su & mask_sta].copy()
        if len(kept):
            kept["source_file"] = Path(fpath).name
            frames.append(kept)

    if not frames:
        raise ValueError(
            "No qualifying events found (pure su AND num_stations > "
            f"{MIN_STATIONS}) across all 2025 CSVs."
        )

    combined = pd.concat(frames, ignore_index=True)
    log.info(
        "Total qualifying events: %d  (pure su, num_stations > %d)",
        len(combined), MIN_STATIONS,
    )
    return combined


def main():
    client = Client(FDSN_CLIENT)

    # ── Collect qualifying events across all daily CSVs ───────────────────
    events = collect_qualifying_events(CSV_DIR)

    # Prepare output columns
    events["enveloc_latitude"]  = np.nan
    events["enveloc_longitude"] = np.nan
    events["n_traces_used"]     = np.nan
    events["location_status"]   = ""

    total = len(events)
    located = 0
    failed  = 0

    for idx, row in events.iterrows():
        event_label = f"[{idx+1}/{total}] cluster {row['cluster_id']} @ {row['rounded_start']}"
        log.info("Processing %s  (%d stations)", event_label, row["num_stations"])

        try:
            stream = download_event_stream(row, client)

            if not stream:
                log.warning("  No waveforms downloaded — skipping.")
                events.at[idx, "location_status"] = "no_waveforms"
                failed += 1
                continue

            lat, lon, n_tr = preprocess_and_locate(stream, client)

            events.at[idx, "n_traces_used"] = n_tr

            if np.isfinite(lat) and np.isfinite(lon):
                events.at[idx, "enveloc_latitude"]  = lat
                events.at[idx, "enveloc_longitude"] = lon
                events.at[idx, "location_status"]   = "located"
                located += 1
                log.info("  ✓  lat=%.4f  lon=%.4f  (%d traces)", lat, lon, n_tr)
            else:
                events.at[idx, "location_status"] = "enveloc_failed"
                failed += 1
                log.warning("  ✗  enveloc returned nan  (%d traces after preprocessing)", n_tr)

        except KeyboardInterrupt:
            log.warning("Interrupted by user — saving partial results.")
            break
        except Exception:
            log.error("  Unexpected error:\n%s", traceback.format_exc())
            events.at[idx, "location_status"] = "error"
            failed += 1

    # ── Save output ───────────────────────────────────────────────────────
    # Drop the raw 'stations' / 'all_classes' list columns from output
    # (they're large strings that aren't needed downstream); keep them if
    # you want by commenting these two lines out.
    out_cols = [
        "cluster_id", "rounded_start", "num_stations",
        "most_common_class", "mean_auc", "mean_max", "mean_prob",
        "enveloc_latitude", "enveloc_longitude",
        "n_traces_used", "location_status", "source_file",
    ]
    # Only keep columns that actually exist (in case CSV schema varies)
    out_cols = [c for c in out_cols if c in events.columns]

    events[out_cols].to_csv(OUTPUT_CSV, index=False)

    log.info("─" * 60)
    log.info("Done.  Located: %d / %d  |  Failed/skipped: %d", located, total, failed)
    log.info("Output saved to: %s", OUTPUT_CSV)


if __name__ == "__main__":
    main()